# Task 1: Financial News Exploratory Data Analysis

Nova Financial Solutions needs a rigorous first pass over the FNSPID financial news dataset before sentiment-return modeling. This notebook profiles headline text, publisher activity, topic patterns, and publication timing so later tasks can separate useful market narrative signals from noise.

## Setup

Place the FNSPID CSV in `../data/raw/`. Expected columns are `headline`, `url`, `publisher`, `date`, and `stock`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_news
from src.eda import (
    daily_article_volume,
    headline_length_summary,
    hourly_article_volume,
    lda_topics,
    publisher_domain_counts,
    top_counts,
    top_tfidf_terms,
)

sns.set_theme(style="whitegrid", palette="Set2")
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
news = load_news(PROJECT_ROOT / "data" / "raw")
news.head()

## 1. Descriptive Statistics

Headline length helps identify whether the dataset is dominated by concise alerts, longer analyst notes, or potential duplicated article metadata. Stock and publisher counts reveal concentration risk in the later sentiment analysis.

In [ ]:
print(f"Rows: {len(news):,}")
print(f"Stocks covered: {news['stock'].nunique():,}")
print(f"Publishers: {news['publisher'].nunique():,}")
print(f"Date range: {news['date'].min()} to {news['date'].max()}")

headline_length_summary(news)

In [ ]:
top_publishers = top_counts(news, "publisher", n=15)
top_stocks = top_counts(news, "stock", n=15)

display(top_publishers.to_frame("article_count"))
display(top_stocks.to_frame("article_count"))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top_publishers.sort_values().plot(kind="barh", ax=ax, color="#4C78A8")
ax.set_title("Top Publishers by Article Count")
ax.set_xlabel("Articles")
ax.set_ylabel("Publisher")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "top_publishers.png", dpi=150)
plt.show()

## 2. Text Analysis: Keywords and Topics

TF-IDF highlights unusually informative words and phrases. LDA provides a lightweight view of repeated themes such as analyst ratings, earnings, clinical milestones, or M&A language.

In [ ]:
terms = top_tfidf_terms(news["headline"], n=25)
terms

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=terms.head(20), y="term", x="score", ax=ax, color="#F58518")
ax.set_title("Top TF-IDF Headline Terms and Phrases")
ax.set_xlabel("Mean TF-IDF score")
ax.set_ylabel("Term")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "top_tfidf_terms.png", dpi=150)
plt.show()

In [ ]:
topics = lda_topics(news["headline"], n_topics=6, n_terms=10)
topics

## 3. Time Series Analysis of News Volume

Publication volume spikes often reflect earnings cycles, major corporate events, analyst-rating waves, or macro shocks. These spikes should be reviewed before treating sentiment as an independent signal.

In [ ]:
daily_volume = daily_article_volume(news)
daily_volume["publication_date"] = pd.to_datetime(daily_volume["publication_date"])

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=daily_volume, x="publication_date", y="article_count", ax=ax, color="#54A24B")
ax.set_title("Daily Financial News Volume")
ax.set_xlabel("Publication date")
ax.set_ylabel("Articles")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "daily_news_volume.png", dpi=150)
plt.show()

daily_volume.sort_values("article_count", ascending=False).head(10)

In [ ]:
hourly_volume = hourly_article_volume(news)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=hourly_volume, x="publication_hour", y="article_count", ax=ax, color="#B279A2")
ax.set_title("Publication Frequency by UTC Hour")
ax.set_xlabel("UTC hour")
ax.set_ylabel("Articles")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "hourly_news_volume.png", dpi=150)
plt.show()

## 4. Publisher Analysis

Publisher concentration matters because repeated language from a small number of sources can bias sentiment scores. Email-like publisher names are also converted to domains to reveal organizational patterns.

In [ ]:
publisher_domains = publisher_domain_counts(news, n=15)
publisher_domains.to_frame("article_count")

In [ ]:
publisher_stock_matrix = (
    news.groupby(["publisher", "stock"])
    .size()
    .rename("article_count")
    .reset_index()
    .sort_values("article_count", ascending=False)
)
publisher_stock_matrix.head(20)

## Initial Insights and Next Steps

Use the tables and charts above to document the highest-volume publishers, dominant tickers, common market-moving phrases, publication-time clustering, and any unusual news spikes. In Task 2, sentiment scores should be joined to trading-day returns using timezone-aware publication dates and adjusted close prices.